In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Edit these for each player

CSV_PATH   = r"data/raw/wc2026_canada_vs_bosnia_and_herzegovina_2026-06-12_events.csv"
MODEL_PATH = r"model/xg_sal313_lr.pkl"
FONT_DIR   = r"assets/fonts"
OUTPUT_DIR = r"."

PLAYER_NAME     = "Son Heung-Min"   # must match exactly as it appears in the CSV
PLAYER_POSITION = "Left Wing"
PLAYER_CLUB     = "LAFC"
PLAYER_ASSISTS  = 0
HEATMAP_HALF    = True   # True = attacking half only, False = full pitch


In [ ]:
import os, glob, pickle, joblib, platform, requests, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.image import imread
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import VerticalPitch
from PIL import Image
from io import BytesIO

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 150

# Fonts
if os.path.exists(FONT_DIR):
    for ttf in glob.glob(os.path.join(FONT_DIR, "**", "*.ttf"), recursive=True):
        fm.fontManager.addfont(ttf)
    fm._load_fontmanager(try_read_cache=False)
    BEBAS  = "Bebas Neue"
    DMSANS = "DM Sans"
else:
    print(f"⚠  Font dir not found: {FONT_DIR}. Download Bebas Neue + DM Sans from Google Fonts.")
    BEBAS  = "DejaVu Sans"
    DMSANS = "DejaVu Sans"

# Model
try:    _obj = joblib.load(MODEL_PATH)
except: _obj = pickle.load(open(MODEL_PATH, "rb"))
_model, _scaler = _obj["model"], _obj["scaler"]
GOAL_Y1, GOAL_Y2, GOAL_Y_CENTER, GOAL_X = 36.0, 44.0, 40.0, 120.0

# Colors
BG         = "#FFFFFF"
PITCH_LINE = "#CCCCCC"
TEXT_DARK  = "#1A1A1A"
TEXT_MID   = "#555555"
TEXT_LIGHT = "#999999"
DIVIDER    = "#E8E8E8"

TEAM_META = {
    "Mexico": ("#CE1126", "mx"), "South Africa": ("#007A4D", "za"),
    "South Korea": ("#C60C30", "kr"), "Republic of Korea": ("#C60C30", "kr"),
    "Korea Republic": ("#C60C30", "kr"), "Czechia": ("#002395", "cz"),
    "Czech Republic": ("#002395", "cz"), "Canada": ("#FF0000", "ca"),
    "Bosnia and Herzegovina": ("#002395", "ba"), "Bosnia & Herzegovina": ("#002395", "ba"),
    "Qatar": ("#8D1B3D", "qa"), "Switzerland": ("#FF0000", "ch"),
    "Brazil": ("#009C3B", "br"), "Morocco": ("#C1272D", "ma"),
    "Haiti": ("#00209F", "ht"), "Scotland": ("#003399", "gb-sct"),
    "United States": ("#B22234", "us"), "USA": ("#B22234", "us"),
    "Paraguay": ("#D52B1E", "py"), "Australia": ("#00008B", "au"),
    "Türkiye": ("#E30A17", "tr"), "Turkey": ("#E30A17", "tr"),
    "Germany": ("#1a1a1a", "de"), "Curaçao": ("#002B7F", "cw"),
    "Curacao": ("#002B7F", "cw"), "Ivory Coast": ("#F77F00", "ci"),
    "Côte d'Ivoire": ("#F77F00", "ci"), "Ecuador": ("#FFD100", "ec"),
    "Netherlands": ("#FF6600", "nl"), "Japan": ("#BC002D", "jp"),
    "Sweden": ("#006AA7", "se"), "Tunisia": ("#E70013", "tn"),
    "Belgium": ("#000000", "be"), "Egypt": ("#C8102E", "eg"),
    "Iran": ("#239F40", "ir"), "IR Iran": ("#239F40", "ir"),
    "New Zealand": ("#00247D", "nz"), "Spain": ("#AA151B", "es"),
    "Cape Verde": ("#003893", "cv"), "Cabo Verde": ("#003893", "cv"),
    "Saudi Arabia": ("#006C35", "sa"), "Uruguay": ("#75AADB", "uy"),
    "France": ("#002395", "fr"), "Senegal": ("#00853F", "sn"),
    "Iraq": ("#CE1126", "iq"), "Norway": ("#EF2B2D", "no"),
    "Argentina": ("#75AADB", "ar"), "Algeria": ("#006233", "dz"),
    "Austria": ("#ED2939", "at"), "Jordan": ("#007A3D", "jo"),
    "Portugal": ("#006600", "pt"), "DR Congo": ("#007FFF", "cd"),
    "Congo DR": ("#007FFF", "cd"), "Uzbekistan": ("#1EB53A", "uz"),
    "Colombia": ("#FCD116", "co"), "England": ("#CF081F", "gb-eng"),
    "Croatia": ("#FF0000", "hr"), "Ghana": ("#EF3340", "gh"),
    "Panama": ("#DA121A", "pa"),
}


In [ ]:
def is_true(v):
    return v is True or str(v).strip().lower() == "true"

def build_xg_features(row):
    px = row["x"] * 1.2
    py = (100 - row["y"]) * 0.8
    dist      = np.sqrt((px - GOAL_X)**2 + (py - GOAL_Y_CENTER)**2)
    angle_rad = abs(np.arctan2(abs(py - GOAL_Y_CENTER), max(GOAL_X - px, 0.1)))
    denom     = px**2 + (py - GOAL_Y_CENTER)**2 - ((GOAL_Y2 - GOAL_Y1) / 2)**2
    vis_angle = max(np.arctan2((GOAL_Y2 - GOAL_Y1) * px, denom), 0)
    return [dist, angle_rad, vis_angle, abs(py - GOAL_Y_CENTER),
            float(is_true(row.get("qual_Head",          False))),
            float(is_true(row.get("qual_BigChance",     False))),
            float(is_true(row.get("qual_FastBreak",     False))),
            float(is_true(row.get("qual_SetPiece",      False)) or
                  is_true(row.get("qual_FreekickTaken", False))),
            float(is_true(row.get("qual_Assisted",      False))),
            0.0,
            float(is_true(row.get("qual_Penalty",       False))),
            float(is_true(row.get("qual_RightFoot",     False))),
            float(is_true(row.get("qual_Head",          False))) * dist,
            float(is_true(row.get("qual_BigChance",     False))) * dist]

def score_shots(shots_df):
    feats    = np.array([build_xg_features(r) for _, r in shots_df.iterrows()])
    X_scaled = _scaler.transform(feats)
    xg       = _model.predict_proba(X_scaled)[:, 1]
    for i, (_, r) in enumerate(shots_df.iterrows()):
        if is_true(r.get("qual_Penalty", False)):
            xg[i] = 0.76
    return xg

def xg_to_size(xg):
    return (18 + (xg - 0.01) / (0.99 - 0.01) * (200 - 18)) * 2

def get_flag(code, size=160):
    url = f"https://flagcdn.com/w{size}/{code}.png"
    try:
        r = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            return Image.open(BytesIO(r.content)).convert("RGBA")
    except: pass
    return None

def get_sofascore_photo(player_name):
    try:
        url = f"https://api.sofascore.com/api/v1/search/all?q={player_name.replace(' ','%20')}"
        r   = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=8)
        for item in r.json().get("results", []):
            if item.get("type") == "player":
                pid   = item["entity"]["id"]
                img_r = requests.get(f"https://img.sofascore.com/api/v1/player/{pid}/image",
                                     headers={"User-Agent":"Mozilla/5.0"}, timeout=8)
                if img_r.status_code == 200:
                    return Image.open(BytesIO(img_r.content)).convert("RGBA")
    except: pass
    return None

def add_image(ax, img, xy, zoom=0.18, zorder=5):
    if img is None: return
    oi = OffsetImage(np.array(img), zoom=zoom)
    ab = AnnotationBbox(oi, xy, frameon=False, zorder=zorder, box_alignment=(0.5,0.5))
    ax.add_artist(ab)


In [ ]:
df          = pd.read_csv(CSV_PATH)
home_team   = df["home_team"].iloc[0]
away_team   = df["away_team"].iloc[0]
home_score  = int(df["home_score"].iloc[0])
away_score  = int(df["away_score"].iloc[0])

_dt        = pd.to_datetime(df["match_date"].iloc[0])
match_date = _dt.strftime("%d %B %Y").lstrip("0") if platform.system() == "Windows"              else _dt.strftime("%-d %B %Y")

player_team  = df[df["player"] == PLAYER_NAME]["team"].iloc[0]
team_color   = TEAM_META.get(player_team, ("#333333","un"))[0]
flag_code    = TEAM_META.get(player_team, ("#333333","un"))[1]
flag_img     = get_flag(flag_code)
player_photo = get_sofascore_photo(PLAYER_NAME)

player_df = df[df["player"] == PLAYER_NAME].copy()
shots     = player_df[player_df["isShot"] == True].copy().reset_index(drop=True)
if len(shots) > 0:
    shots["xg"] = score_shots(shots)
else:
    shots["xg"] = pd.Series(dtype=float)

on_tgt   = shots[
    shots["event"].isin(["SavedShot","Goal"]) &
    (~shots.get("qual_Blocked", pd.Series(False, index=shots.index)).astype(bool))
]
touches  = player_df[player_df["isTouch"] == True].dropna(subset=["x","y"]).copy()

n_shots   = len(shots)
n_on_tgt  = len(on_tgt)
n_goals   = len(shots[shots["event"] == "Goal"])
total_xg  = round(shots["xg"].sum(), 2) if len(shots) > 0 else 0.0
score_str = f"{home_team} {home_score}–{away_score} {away_team}"

print(f"Player: {PLAYER_NAME}  ({player_team})")
print(f"Shots: {n_shots}  |  On target: {n_on_tgt}  |  Goals: {n_goals}  |  xG: {total_xg}")
print(f"Touches: {len(touches)}")


In [ ]:
FIG_W, FIG_H = 1600/150, 900/150
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor=BG)

HEADER_H  = 0.20
BODY_TOP  = 1.0 - HEADER_H
PITCH_BOT = 0.08
PITCH_H   = BODY_TOP - PITCH_BOT - 0.03
PITCH_W   = 0.46
GAP       = 0.04
L_PITCH   = 0.02
R_PITCH   = L_PITCH + PITCH_W + GAP

# Header
ax_hdr = fig.add_axes([0, BODY_TOP, 1, HEADER_H])
ax_hdr.set_facecolor(BG); ax_hdr.set_xlim(0,1); ax_hdr.set_ylim(0,1); ax_hdr.axis("off")
ax_hdr.axhline(0.08, color=DIVIDER, linewidth=1.2, xmin=0.01, xmax=0.99)
ax_hdr.add_patch(mpatches.Rectangle((0,0),0.007,1.0,facecolor=team_color,edgecolor="none",
                                      transform=ax_hdr.transAxes))
add_image(ax_hdr, flag_img,     (0.052, 0.55), zoom=0.20)
add_image(ax_hdr, player_photo, (0.052, 0.18), zoom=0.12)

ax_hdr.text(0.088, 0.80, player_team.upper(), ha="left", va="center",
            fontfamily=BEBAS, fontsize=11, color=team_color, transform=ax_hdr.transAxes)
ax_hdr.text(0.088, 0.55, PLAYER_NAME.upper(), ha="left", va="center",
            fontfamily=BEBAS, fontsize=30, color=TEXT_DARK, transform=ax_hdr.transAxes)
ax_hdr.text(0.088, 0.30, f"{PLAYER_POSITION}  ·  {PLAYER_CLUB}", ha="left", va="center",
            fontfamily=DMSANS, fontsize=9, color=TEXT_LIGHT, transform=ax_hdr.transAxes)
ax_hdr.text(0.088, 0.14, f"{score_str}  ·  {match_date}", ha="left", va="center",
            fontfamily=DMSANS, fontsize=8, color=TEXT_LIGHT, transform=ax_hdr.transAxes)

STATS = [("SHOTS",n_shots),("ON TARGET",n_on_tgt),(f"xG",f"{total_xg:.2f}"),
         ("GOALS",n_goals),("ASSISTS",PLAYER_ASSISTS)]
stat_start_x = 0.39; stat_spacing = 0.118
for k, (label, val) in enumerate(STATS):
    sx = stat_start_x + k * stat_spacing
    ax_hdr.text(sx, 0.60, str(val), ha="center", va="center",
                fontfamily=BEBAS, fontsize=28, color=team_color, transform=ax_hdr.transAxes)
    ax_hdr.text(sx, 0.24, label, ha="center", va="center",
                fontfamily=DMSANS, fontsize=7.5, fontweight="600", color=TEXT_LIGHT,
                transform=ax_hdr.transAxes)
    if k < len(STATS)-1:
        div_x = sx + stat_spacing/2
        ax_hdr.plot([div_x,div_x],[0.12,0.90],color=DIVIDER,linewidth=0.8,
                    transform=ax_hdr.transAxes)

# Shot map (left)
ax_shot = fig.add_axes([L_PITCH, PITCH_BOT, PITCH_W, PITCH_H])
pitch_shot = VerticalPitch(pitch_type="opta", pitch_color=BG, line_color=PITCH_LINE,
                           linewidth=1.2, half=True, goal_type="box",
                           corner_arcs=True, line_zorder=4)
pitch_shot.draw(ax=ax_shot); ax_shot.set_facecolor(BG)

for _, s in shots.iterrows():
    size = xg_to_size(s["xg"])
    if s["event"] == "Goal":
        ax_shot.scatter(s["y"], s["x"], s=size, c=team_color, alpha=1.0,
                        edgecolors=team_color, linewidths=0.5, marker="o", zorder=5)
    else:
        ax_shot.scatter(s["y"], s["x"], s=size, facecolors="none",
                        edgecolors=team_color, linewidths=1.4, marker="o", zorder=5, alpha=0.85)

ax_shot.set_title("SHOT MAP", fontfamily=BEBAS, fontsize=13, color=TEXT_MID, pad=8)
ax_shot.scatter(0.18,-0.05,s=50,facecolors="none",edgecolors=team_color,linewidths=1.4,
                marker="o",zorder=6,transform=ax_shot.transAxes,clip_on=False)
ax_shot.text(0.22,-0.05,"Shot (not goal)",ha="left",va="center",fontfamily=DMSANS,
             fontsize=7.5,color=TEXT_MID,transform=ax_shot.transAxes,clip_on=False)
ax_shot.scatter(0.55,-0.05,s=50,c=team_color,edgecolors=team_color,linewidths=0.5,
                marker="o",zorder=6,transform=ax_shot.transAxes,clip_on=False)
ax_shot.text(0.59,-0.05,"Goal",ha="left",va="center",fontfamily=DMSANS,
             fontsize=7.5,color=TEXT_MID,transform=ax_shot.transAxes,clip_on=False)

         fontfamily=DMSANS, fontsize=9.4, fontweight="bold",
         color=TEXT_DARK, va="center", ha="left")

# Touch heatmap (right)
ax_heat = fig.add_axes([R_PITCH, PITCH_BOT, PITCH_W, PITCH_H])
pitch_heat = VerticalPitch(pitch_type="opta", pitch_color=BG, line_color=PITCH_LINE,
                           linewidth=1.2, half=HEATMAP_HALF, goal_type="box",
                           corner_arcs=True, line_zorder=4)
ax_heat.set_facecolor(BG); pitch_heat.draw(ax=ax_heat)

if len(touches) >= 5:
    pitch_heat.kdeplot(touches["y"], touches["x"], ax=ax_heat,
                       cmap="coolwarm", fill=True, levels=100,
                       thresh=0, cut=10, zorder=2, alpha=0.75)

ax_heat.scatter(touches["y"], touches["x"], s=6, c="white", alpha=0.20, zorder=5)
ax_heat.set_title(f"TOUCH MAP  ({len(touches)} touches)",
                  fontfamily=BEBAS, fontsize=13, color=TEXT_MID, pad=8)

# Save
safe_p   = PLAYER_NAME.lower().replace(" ","_")
date_str = df["match_date"].iloc[0]
out_path = os.path.join(OUTPUT_DIR, f"wc_player_{safe_p}_{date_str}.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=BG, edgecolor="none")
print(f"✓ Saved → {out_path}")
plt.show()
